In [1]:
import requests
import time
import pandas as pd
from dotenv import load_dotenv
import os

# === Load token from .env ===
ENV_FILE = 'All_tokens.env'
load_dotenv(ENV_FILE)
token = os.getenv('GITHUB_TOKEN')

if not token:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

headers = {"Authorization": f"token {token.strip()}"}

# === Define the single project you want to search ===
Searching_Project = "cboy"  # 🔍 Your search keyword

# === Rate Limit Checker ===
def check_rate_limit():
    url = "https://api.github.com/rate_limit"
    response = requests.get(url, headers=headers)
    data = response.json()
    remaining = data['rate']['remaining']
    limit = data['rate']['limit']
    print(f"⏳ GitHub API: {remaining} / {limit} requests remaining.")
    return remaining

# === GitHub Search Function ===
def search_github_repo(project_name):
    query = f"{project_name} in:name"
    url = f"https://api.github.com/search/repositories?q={query}&sort=stars&order=desc&per_page=1"

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()

        if data.get("total_count", 0) > 0:
            return data["items"][0]["html_url"]
        else:
            return ""
    except Exception as e:
        print(f"❌ Error for {project_name}: {e}")
        return ""

# === Run the single project search ===
check_rate_limit()
url = search_github_repo(Searching_Project)

# === Save to CSV ===
df_out = pd.DataFrame([{'project': Searching_Project, 'github_url': url}])
df_out.to_csv("Single_Project_GitHub_URL.csv", index=False)

print("✅ Result:")
print(df_out)


⏳ GitHub API: 5000 / 5000 requests remaining.
✅ Result:
  project                        github_url
0    cboy  https://github.com/jkbenaim/cboy
